<a href="https://colab.research.google.com/github/pinkadace-code/llm-context-generator/blob/main/llm_data_visualizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Šūna 1: Instalācija
!pip install mysql-connector-python groq pandas matplotlib seaborn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.7 MB/s eta 0:00:00


In [2]:
# Šūna 2: Imports un konfigurācija
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Colab vajadzīgs
import seaborn as sns
import json
import re
import base64
import io
from groq import Groq
from google.colab import userdata

# Groq klients
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# MySQL savienojums
def get_connection():
    return mysql.connector.connect(
        host="87.110.123.151",
        port=3306,
        user="fita",
        database="direct_payments"
    )

# Konteksts (no 1. uzdevuma — datubāzes struktūra)
DB_CONTEXT = """
Datubāze: direct_payments

Tabulas:
1. organisations
   - id (varchar): unikālais identifikators
   - created_at (datetime): izveides datums
   - parent_vertical (varchar): nozare/kategorija

2. mandates
   - id (varchar): unikālais identifikators
   - created_at (datetime): izveides datums
   - scheme (varchar): maksājumu shēma (BACS, SEPA u.c.)
   - organisation_id (varchar): → organisations.id

3. payments
   - id (varchar): unikālais identifikators
   - amount (double): summa
   - currency (varchar): valūta
   - created_at (datetime): izveides datums
   - source (varchar): avots
   - charge_date (datetime): iekasēšanas datums
   - mandate_id (varchar): → mandates.id

Saistības: organisations → mandates → payments
"""

print("✅ Konfigurācija gatava!")

✅ Konfigurācija gatava!


In [3]:
# Šūna 3: LLM ģenerē vizualizācijas plānu

def generate_plan(context):
    prompt = f"""
Tu esi datu analītiķis. Tev ir šāda datubāze:

{context}

Izveido vizualizācijas plānu JSON formātā. Atgriez TIKAI JSON, bez papildus teksta.

JSON struktūra:
{{
  "plan": [
    {{
      "id": 1,
      "title": "Grafika nosaukums",
      "description": "Ko šis grafiks parāda",
      "chart_type": "bar|line|pie|scatter|heatmap",
      "sql_hint": "Apraksts kādi dati vajadzīgi SQL vaicājumam"
    }},
    ...
  ]
}}

Izveido 5-7 dažādus, interesantus vizuāļus kas atklāj ieskatus par maksājumiem, organizācijām un mandātiem.
Izmanto dažādus grafiku tipus. Katram grafika punktam pievieno skaidru sql_hint.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    raw = response.choices[0].message.content

    # Izņem JSON no atbildes
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        plan = json.loads(json_match.group())
        return plan
    else:
        raise ValueError(f"Nevarēja parsēt JSON:\n{raw}")

# Ģenerē plānu
print("🤖 Ģenerēju vizualizācijas plānu...")
plan = generate_plan(DB_CONTEXT)

print("\n📋 ĢENERĒTAIS PLĀNS:\n")
print("=" * 60)
for item in plan["plan"]:
    print(f"\n[{item['id']}] {item['title']}")
    print(f"    Grafika tips: {item['chart_type']}")
    print(f"    Apraksts: {item['description']}")
    print(f"    SQL hints: {item['sql_hint']}")
    print("-" * 60)

🤖 Ģenerēju vizualizācijas plānu...

📋 ĢENERĒTAIS PLĀNS:


[1] Maksājumu summa pa organizācijām
    Grafika tips: bar
    Apraksts: Parāda kopējo maksājumu summu pa organizācijām
    SQL hints: SELECT SUM(p.amount) AS total_amount, o.id AS organisation_id FROM payments p JOIN mandates m ON p.mandate_id = m.id JOIN organisations o ON m.organisation_id = o.id GROUP BY o.id
------------------------------------------------------------

[2] Maksājumu skaits pa maksājumu shēmām
    Grafika tips: pie
    Apraksts: Parāda maksājumu skaitu pa dažādām maksājumu shēmām
    SQL hints: SELECT COUNT(p.id) AS payment_count, m.scheme AS scheme FROM payments p JOIN mandates m ON p.mandate_id = m.id GROUP BY m.scheme
------------------------------------------------------------

[3] Maksājumu summa pa mēnešiem
    Grafika tips: line
    Apraksts: Parāda maksājumu summu pa mēnešiem
    SQL hints: SELECT SUM(p.amount) AS total_amount, EXTRACT(MONTH FROM p.charge_date) AS month FROM payments p GROUP BY EXTRA

In [4]:
# Šūna 4: SQL ģenerēšana un izpilde

def generate_sql(plan_item, context):
    prompt = f"""
Datubāze:
{context}

Uzdevums: {plan_item['description']}
SQL hints: {plan_item['sql_hint']}

Uzraksti MySQL SQL vaicājumu kas iegūs datus šim vizuālim.
Atgriez TIKAI SQL vaicājumu, bez paskaidrojumiem.
Izmanto tikai tabulas: organisations, mandates, payments.
Neizmanto subqueries ja nav nepieciešams.
Ierobežo rezultātus ar LIMIT 100 ja vajadzīgs.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    sql = response.choices[0].message.content.strip()

    # Notīra SQL no markdown
    sql = re.sub(r'```sql\n?', '', sql)
    sql = re.sub(r'```\n?', '', sql)
    return sql.strip()


def validate_and_fetch(sql):
    """Validē SQL un iegūst datus"""
    # Drošības pārbaude — tikai SELECT
    if not sql.strip().upper().startswith("SELECT"):
        raise ValueError("Drošības kļūda: tikai SELECT atļauts!")

    conn = get_connection()
    try:
        df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        raise Exception(f"SQL kļūda: {e}\nSQL: {sql}")
    finally:
        conn.close()


def process_plan_item(plan_item):
    """Apstrādā vienu plāna punktu: SQL → dati → grafiks → apraksts"""
    print(f"\n🔄 Apstrādāju: {plan_item['title']}")

    # 1. SQL ģenerēšana
    sql = generate_sql(plan_item, DB_CONTEXT)
    print(f"   SQL: {sql[:80]}...")

    # 2. Datu iegūšana
    df = validate_and_fetch(sql)
    print(f"   ✅ Iegūtas {len(df)} rindas, {len(df.columns)} kolonnas")

    return sql, df

print("✅ Funkcijas definētas!")

✅ Funkcijas definētas!


In [14]:
# Šūna 5: Vizualizācijas funkcija

def create_chart(plan_item, df):
    fig, ax = plt.subplots(figsize=(10, 5))
    chart_type = plan_item['chart_type']
    cols = df.columns.tolist()

    try:
        # Automātiski nosaka kurā kolonnā ir skaitļi, kurā teksts
        numeric_cols = df.select_dtypes(include='number').columns.tolist()
        text_cols = df.select_dtypes(exclude='number').columns.tolist()

        if chart_type == "bar":
            x_col = text_cols[0] if text_cols else cols[0]
            y_col = numeric_cols[0] if numeric_cols else cols[1]
            df_plot = df.head(15)
            ax.bar(df_plot[x_col].astype(str), df_plot[y_col], color='steelblue')
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
            plt.xticks(rotation=45, ha='right')

        elif chart_type == "pie":
            label_col = text_cols[0] if text_cols else cols[0]
            value_col = numeric_cols[0] if numeric_cols else cols[1]
            df_plot = df.head(8)
            ax.pie(df_plot[value_col], labels=df_plot[label_col].astype(str),
                   autopct='%1.1f%%', startangle=90)

        elif chart_type == "line":
            x_col = text_cols[0] if text_cols else cols[0]
            y_col = numeric_cols[0] if numeric_cols else cols[1]
            ax.plot(df[x_col].astype(str), df[y_col], marker='o', color='coral')
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
            plt.xticks(rotation=45, ha='right')

        elif chart_type == "scatter":
            x_col = numeric_cols[0] if len(numeric_cols) > 0 else cols[0]
            y_col = numeric_cols[1] if len(numeric_cols) > 1 else cols[1]
            ax.scatter(df[x_col], df[y_col], alpha=0.6, color='purple')
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)

        elif chart_type == "heatmap":
            numeric_df = df.select_dtypes(include='number')
            if not numeric_df.empty:
                sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f',
                           cmap='coolwarm', ax=ax)
        else:
            x_col = text_cols[0] if text_cols else cols[0]
            y_col = numeric_cols[0] if numeric_cols else cols[1]
            ax.bar(df[x_col].astype(str).head(10), df[y_col].head(10))
            plt.xticks(rotation=45, ha='right')

        ax.set_title(plan_item['title'], fontsize=14, fontweight='bold', pad=15)
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=120, bbox_inches='tight')
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode('utf-8')
        plt.close()
        return img_b64

    except Exception as e:
        plt.close()
        raise Exception(f"Grafika kļūda: {e}")

print("✅ Vizualizācijas funkcija gatava!")

✅ Labotā funkcija gatava!


In [6]:
# Šūna 6: AI apraksts

def generate_insight(plan_item, df):
    """LLM ģenerē aprakstu un ieskatus par datiem"""
    # Datu kopsavilkums LLM
    data_summary = df.describe(include='all').to_string()
    sample = df.head(5).to_string()

    prompt = f"""
Tu esi datu analītiķis. Tev ir šādi dati vizuālim "{plan_item['title']}":

Datu parauga (5 rindas):
{sample}

Statistika:
{data_summary}

Uzraksti īsu analīzi latviešu valodā (3-4 teikumi):
1. Ko šie dati parāda?
2. Kāds ir galvenais atklājums?
3. Ko varētu uzlabot vai izpētīt tālāk?

Esi konkrēts, izmanto skaitļus no datiem.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4
    )
    return response.choices[0].message.content.strip()

print("✅ Apraksta funkcija gatava!")

✅ Apraksta funkcija gatava!


In [7]:
# Šūna 7: HTML lapas izveide

def generate_html_report(results):
    """Izveido HTML lapu ar visiem vizuāļiem"""

    cards_html = ""
    for i, r in enumerate(results):
        cards_html += f"""
        <div class="card">
            <div class="card-header">
                <span class="card-num">{i+1}</span>
                <div>
                    <h2>{r['title']}</h2>
                    <span class="badge">{r['chart_type']}</span>
                </div>
            </div>
            <img src="data:image/png;base64,{r['image']}" alt="{r['title']}"/>
            <div class="insight">
                <h3>💡 Ieskats</h3>
                <p>{r['insight']}</p>
            </div>
            <details>
                <summary>🔍 SQL vaicājums</summary>
                <code>{r['sql']}</code>
            </details>
        </div>
        <div class="divider">◆ ◆ ◆</div>
        """

    html = f"""<!DOCTYPE html>
<html lang="lv">
<head>
<meta charset="UTF-8">
<title>Datu Vizualizācijas Pārskats</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Segoe UI', sans-serif; background: #f0f2f5; color: #222; }}
  header {{ background: linear-gradient(135deg, #1a1a2e, #16213e);
            color: white; padding: 40px; text-align: center; }}
  header h1 {{ font-size: 2.2em; letter-spacing: 2px; }}
  header p {{ margin-top: 8px; opacity: 0.8; font-size: 1.1em; }}
  .container {{ max-width: 960px; margin: 40px auto; padding: 0 20px; }}
  .card {{ background: white; border-radius: 16px; padding: 30px;
           box-shadow: 0 4px 20px rgba(0,0,0,0.08); margin-bottom: 20px; }}
  .card-header {{ display: flex; align-items: center; gap: 16px; margin-bottom: 20px; }}
  .card-num {{ background: #1a1a2e; color: white; width: 42px; height: 42px;
               border-radius: 50%; display: flex; align-items: center;
               justify-content: center; font-weight: bold; font-size: 1.1em; flex-shrink: 0; }}
  .card h2 {{ font-size: 1.3em; color: #1a1a2e; }}
  .badge {{ display: inline-block; background: #e8f0fe; color: #1a73e8;
            padding: 3px 12px; border-radius: 20px; font-size: 0.8em;
            margin-top: 5px; text-transform: uppercase; letter-spacing: 1px; }}
  .card img {{ width: 100%; border-radius: 10px; margin: 10px 0; }}
  .insight {{ background: #f8f9ff; border-left: 4px solid #1a73e8;
              padding: 15px 20px; border-radius: 0 10px 10px 0; margin: 15px 0; }}
  .insight h3 {{ color: #1a73e8; margin-bottom: 8px; }}
  .insight p {{ line-height: 1.6; color: #444; }}
  details {{ margin-top: 12px; }}
  summary {{ cursor: pointer; color: #666; font-size: 0.9em; padding: 5px 0; }}
  code {{ display: block; background: #1e1e1e; color: #d4d4d4; padding: 15px;
          border-radius: 8px; font-size: 0.85em; margin-top: 8px;
          white-space: pre-wrap; word-break: break-all; }}
  .divider {{ text-align: center; color: #ccc; font-size: 1.2em;
              margin: 10px 0 30px 0; letter-spacing: 8px; }}
  footer {{ text-align: center; padding: 30px; color: #999; font-size: 0.9em; }}
</style>
</head>
<body>
<header>
  <h1>📊 Datu Vizualizācijas Pārskats</h1>
  <p>Direct Payments datubāze — ģenerēts ar LLM</p>
</header>
<div class="container">
  {cards_html}
</div>
<footer>Ģenerēts automātiski ar Groq LLM (llama-3.3-70b-versatile)</footer>
</body>
</html>"""
    return html

print("✅ HTML funkcija gatava!")

✅ HTML funkcija gatava!


In [15]:
# Šūna 8: GALVENĀ IZPILDE

results = []

for plan_item in plan["plan"]:
    try:
        # SQL + dati
        sql, df = process_plan_item(plan_item)

        # Grafiks
        img_b64 = create_chart(plan_item, df)

        # AI apraksts
        insight = generate_insight(plan_item, df)
        print(f"   💬 Apraksts ģenerēts")

        results.append({
            "title": plan_item["title"],
            "chart_type": plan_item["chart_type"],
            "sql": sql,
            "image": img_b64,
            "insight": insight
        })
        print(f"   ✅ Pabeigts!")

    except Exception as e:
        print(f"   ❌ Kļūda: {e}")

# HTML izveide
html = generate_html_report(results)
with open("report.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"\n🎉 PABEIGTS! {len(results)} no {len(plan['plan'])} vizuāļi veiksmīgi!")
print("📄 Fails: report.html")


🔄 Apstrādāju: Maksājumu summa pa organizācijām
   SQL: SELECT 
  SUM(p.amount) AS total_amount, 
  o.id AS organisation_id
FROM 
  paym...


/tmp/ipykernel_3747/110860385.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


   ✅ Iegūtas 100 rindas, 2 kolonnas
   💬 Apraksts ģenerēts
   ✅ Pabeigts!

🔄 Apstrādāju: Maksājumu skaits pa maksājumu shēmām
   SQL: SELECT 
  COUNT(p.id) AS payment_count, 
  m.scheme AS scheme 
FROM 
  payments ...


/tmp/ipykernel_3747/110860385.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


   ✅ Iegūtas 2 rindas, 2 kolonnas
   💬 Apraksts ģenerēts
   ✅ Pabeigts!

🔄 Apstrādāju: Maksājumu summa pa mēnešiem
   SQL: SELECT 
  SUM(p.amount) AS total_amount, 
  EXTRACT(MONTH FROM p.charge_date) AS...


...

   ✅ Iegūtas 2 rindas, 2 kolonnas
   💬 Apraksts ģenerēts
   ✅ Pabeigts!

🔄 Apstrādāju: Maksājumu skaits pa organizāciju izveides datumiem
   SQL: SELECT 
  COUNT(p.id) AS payment_count, 
  EXTRACT(YEAR FROM o.created_at) AS ye...


/tmp/ipykernel_3747/110860385.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


   ✅ Iegūtas 4 rindas, 3 kolonnas
   💬 Apraksts ģenerēts
   ✅ Pabeigts!

🔄 Apstrādāju: Maksājumu summa pa avotiem
   SQL: SELECT 
  SUM(p.amount) AS total_amount, 
  p.source AS source 
FROM 
  payments...


/tmp/ipykernel_3747/110860385.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


   ✅ Iegūtas 3 rindas, 2 kolonnas
   💬 Apraksts ģenerēts
   ✅ Pabeigts!

🎉 PABEIGTS! 7 no 7 vizuāļi veiksmīgi!
📄 Fails: report.html


In [16]:
# Šūna 9: Lejupielāde un GitHub

# Lejupielādē HTML failu
from google.colab import files
files.download('report.html')

# Saglabā notebook GitHub
# Colab: File → Download → Download .ipynb
# Tad augšupielādē savā GitHub repo
print("📥 Lejupielādē report.html")
print("📓 Saglabā notebook: File → Download → .ipynb")
print("🔗 Augšupielādē abus failus GitHub repozitorijā")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Lejupielādē report.html
📓 Saglabā notebook: File → Download → .ipynb
🔗 Augšupielādē abus failus GitHub repozitorijā
